https://chatgpt.com/g/g-p-68d161e5e43c81918b404ba2411ad90e-slk-correlaid/c/68fe80cf-0180-832a-9a22-e1ba939d400a

In [84]:
%pip install requests tqdm
%pip install hda -U

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [85]:
import os
import sys
from pathlib import Path
from typing import Dict, List, Tuple

from hda import Client, Configuration
from tqdm import tqdm


In [ ]:
# Kosovo bbox in EPSG:4326 [minLon, minLat, maxLon, maxLat]
BBOX = [20.0, 41.8, 21.8, 43.3]

# Output root
# OUT_DIR = Path("/Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/.shortcut-targets-by-id/1p4PDClIbV5Br-EkGzuvhH5WVU1BudYkH/SLK CorrelAid 2025/Copernicus/batchimport")
OUT_DIR = Path("/Users/vlad/Downloads/hda_test_hrl_tc")

# Datasets and their temporal parameters
# For CHANGE products, HRL uses periods like "2012-2015", "2015-2018", "2018-2023".
DATASETS: Dict[str, Dict] = {
    # Tree Cover Density (numeric % per pixel)
    "EO:EEA:DAT:HRL:TCF": {
        "label": "TCD",
        "years": ["2012", "2015", "2018", "2019", "2020","2021","2022", "2023"],
        "extra": {"resolution": "10m", "productType": "Tree Cover Density", "itemsPerPage": 200, "startIndex": 0}
    },
    # Confidence layer for TCD
    "EO:EEA:DAT:HRL:TCF": {
        "label": "TCD_CONF",
        "years": ["2012", "2015", "2018", "2019", "2020","2021","2022", "2023"],
        "extra": {"resolution": "10m", "productType": "Tree Cover Density Confidence Layer"}
    },
    # Tree Cover Presence Change
    "EO:EEA:DAT:HRL:TCF": {
        "label": "TCD_CHANGE",
        "periods": ["2012-2015", "2015-2018", "2018-2021"],
        "extra": {"resolution": "20m", "productType": "Tree Cover Presence Change"}
    },
    # Confidence layer for Change
    "EO:EEA:DAT:HRL:TCF": {
        "label": "TCD_CHANGE_CONF",
        "periods": ["2012-2015", "2015-2018", "2018-2021"],
        "extra": {"resolution": "20m", "productType": "Tree Cover Presence Change Confidence Layer"}
    },
}

# HDA pagination
ITEMS_PER_PAGE = 200
START_INDEX = 0

# Toggle automatic merge with GDAL after download (per group)
RUN_GDAL_MERGE = False  # set to True if you want automatic mosaics
GDAL_OUTPUT_CRS = "EPSG:3857"  # used only in the example warp step below

In [87]:
def ensure_outdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

In [88]:
def hda_client_or_exit() -> Client:
    hdarc = Path(Path.home()/'.hdarc')
    if not hdarc.is_file():
        import getpass
        USERNAME = input('Enter your username: ')
        PASSWORD = getpass.getpass('Enter your password: ')

        with open(Path.home()/'.hdarc', 'w') as f:
            f.write(f'user: {USERNAME}\n')
            f.write(f'password:{PASSWORD}\n')
    else:
        print('Configuration file already exists.')

    try:
        client = Client()
        client.metadata(dataset_id="EO:EEA:DAT:HRL:TCF")  # Test connection
        print(f"Logged in as {client.config.user}")
        return client
    except Exception as e:
        print("Failed to initialize HDA Client. Check your .hdarc file.")
        raise

In [89]:
def page_search(client: Client, query: dict) -> list:
    print(f"Searching {query['dataset_id']} with filter {query}")
    try:
        results = client.search(
            dataset_id=query["dataset_id"],            
            productType=query.get("productType"),
            resolution=query.get("resolution"),
            year=query.get("year"),
            bbox=query["bbox"],
            itemsPerPage=query.get("itemsPerPage"),
            startIndex=query.get("startIndex")
        )
        print(f"  → {len(results)} products found")
        return results
    except Exception as e:
        print(f"Error during search: {e}")
        return []

In [90]:
def download_matches(matches: list, out_dir: Path):
    ensure_outdir(out_dir)
    for prd in tqdm(matches, desc=f"Downloading to {out_dir}", unit="tile"):
        try:
            prd.download(str(out_dir))
        except Exception as e:
            print(f"Download failed for {getattr(prd, 'id', 'unknown')}: {e}")

In [91]:
def gdal_merge_group(in_dir: Path, pattern: str, out_path: Path):
    import subprocess, glob
    files = sorted(glob.glob(str(in_dir / pattern)))
    if not files:
        print(f"No files to merge for pattern {pattern} in {in_dir}")
        return
    ensure_outdir(out_path.parent)
    cmd = ["gdal_merge.py", "-o", str(out_path), "-of", "GTiff", "-a_nodata", "0"] + files
    subprocess.run(cmd, check=True)

In [92]:
hda_client = Client()
help(hda_client.metadata)

print(f"Logged in as {hda_client.config.user}")

Help on method metadata in module hda.api:

metadata(dataset_id) method of hda.api.Client instance
    Returns the metadata object for the given dataset.
    
    :param dataset_id: The dataset ID
    :type dataset_id: str

Logged in as vladcorrelaid


In [93]:
hda_client.metadata(dataset_id="EO:EEA:DAT:HRL:TCF")  # Test connection
# hda_client.metadata(dataset_id="EO:EEA:DAT:CLMS_HRVPP_VPP")  # Test connection

{'type': 'object',
 'title': 'Queryable',
 'properties': {'dataset_id': {'title': 'Dataset_id',
   'type': 'string',
   'oneOf': [{'const': 'EO:EEA:DAT:HRL:TCF',
     'title': 'EO:EEA:DAT:HRL:TCF',
     'group': None}]},
  'bbox': {'title': 'bbox',
   'type': 'array',
   'minItems': 4,
   'maxItems': 4,
   'items': [{'type': 'number', 'maximum': 180, 'minimum': -180},
    {'type': 'number', 'maximum': 90, 'minimum': -90},
    {'type': 'number', 'maximum': 180, 'minimum': -180},
    {'type': 'number', 'maximum': 90, 'minimum': -90}]},
  'productType': {'title': 'Product Type',
   'type': 'string',
   'oneOf': [{'const': 'Broadleaved Cover Density',
     'title': 'Broadleaved Cover Density',
     'group': None},
    {'const': 'Coniferous Cover Density',
     'title': 'Coniferous Cover Density',
     'group': None},
    {'const': 'Dominant Leaf Type',
     'title': 'Dominant Leaf Type',
     'group': None},
    {'const': 'Dominant Leaf Type Change',
     'title': 'Dominant Leaf Type Chang

In [96]:
ensure_outdir(OUT_DIR)
client = hda_client_or_exit()

# Test with one dataset
dataset_id = "EO:EEA:DAT:HRL:TCF"  # Example: Tree Cover Density
cfg = DATASETS[dataset_id]
label = "TCD_CONF"
#periods 20m TCD_CHANGE - Tree Cover Presence Change
#periods 20m TCD_CHANGE_CONF - Tree Cover Presence Change Confidence Layer
years = [2012, 2013, 2015, 2018, 2019, 2020, 2021, 2022, 2023]
periods = ["2012-2015", "2015-2018", "2018-2021"]

# Iterate through all years in the dataset configuration
for year in years:
    # print(f"\n=== Testing {label} {year} ===")
    print(year)
    out = OUT_DIR / label / str(year)
    ensure_outdir(out)

    query = {
        "dataset_id": dataset_id,  # Required
        "productType": "Tree Cover Density Confidence Layer",  # Matches allowed values
        "resolution": "10m",  # Matches allowed values
        "year": str(year),  # Matches allowed values
        "bbox": [20.0, 41.8, 21.8, 43.3],  # Valid bbox
        "itemsPerPage": 200,  # Matches pattern
        "startIndex": 0  # Matches pattern
    }

    # Search and download products for the current year
    products = client.search(query)
    download_matches(products, out)

print("\n✅ Test complete. Files saved under:", OUT_DIR.resolve())

Configuration file already exists.
Logged in as vladcorrelaid
2012


2013


2015


2018


2019


2020


2021


2022


2023



✅ Test complete. Files saved under: /Users/vlad/Downloads/hda_test_hrl_tc


In [ ]:
ensure_outdir(OUT_DIR)
client = hda_client_or_exit()

# Test with one dataset
dataset_id = "EO:EEA:DAT:HRL:TCF"  # Example: Tree Cover Density
cfg = DATASETS[dataset_id]
label = cfg["label"]
base_extra = cfg.get("extra", {})

# Iterate through all years in the dataset configuration
for year in cfg["years"]:
    print(f"\n=== Testing {label} {year} ===")
    out = OUT_DIR / label / year
    ensure_outdir(out)

    query = {
        "dataset_id": dataset_id,  # Required
        "productType": "Tree Cover Density Confidence Layer",  # Matches allowed values
        "resolution": "10m",  # Matches allowed values
        "year": year,  # Matches allowed values
        "bbox": [20.0, 41.8, 21.8, 43.3],  # Valid bbox
        "itemsPerPage": 200,  # Matches pattern
        "startIndex": 0  # Matches pattern
    }

    # Search and download products for the current year
    products = client.search(query)
    download_matches(products, out)

print("\n✅ Test complete. Files saved under:", OUT_DIR.resolve())

In [ ]:
ensure_outdir(OUT_DIR)
client = hda_client_or_exit()

# Test with one dataset
dataset_id = "EO:EEA:DAT:HRL:TCF"  # Example: Tree Cover Density
cfg = DATASETS[dataset_id]
label = cfg["label"]
base_extra = cfg.get("extra", {})

# Test with one year
year = "2023"  # Example: Year 2012
print(f"\n=== Testing {label} {year} ===")
out = OUT_DIR / label / year
ensure_outdir(out)

# query = {
#     "dataset_id": dataset_id,
#      "bbox": BBOX,
#      "year": year     
#     }
# query.update(base_extra)
query = {
    "dataset_id": "EO:EEA:DAT:HRL:TCF",  # Required
    "productType": "Tree Cover Density",  # Matches allowed values
    "resolution": "10m",  # Matches allowed values
    "year": year,  # Matches allowed values
    "bbox": [20.0, 41.8, 21.8, 43.3],  # Valid bbox
    "itemsPerPage": 200,  # Matches pattern
    "startIndex": 0  # Matches pattern
}
# products = page_search(client, query)
products = client.search(query)
download_matches(products, out)


print("\n✅ Test complete. Files saved under:", OUT_DIR.resolve())

Configuration file already exists.
Logged in as vladcorrelaid

=== Testing TCD 2023 ===



✅ Test complete. Files saved under: /Users/vlad/Downloads/hda_test_hrl_tc


In [ ]:
def main():
    ensure_outdir(OUT_DIR)
    client = hda_client_or_exit()

    for dataset_id, cfg in DATASETS.items():
        label = cfg["label"]
        base_extra = cfg.get("extra", {})

        # Yearly datasets
        for year in cfg.get("years", []):
            print(f"\n=== {label} {year} ===")
            out = OUT_DIR / label / year
            ensure_outdir(out)
            query = {"dataset_id": dataset_id, "bbox": BBOX, "year": year}
            query.update(base_extra)
            products = page_search(client, query)
            download_matches(products, out)

        # Period datasets
        for period in cfg.get("periods", []):
            print(f"\n=== {label} {period} ===")
            out = OUT_DIR / label / period
            ensure_outdir(out)
            query = {"dataset_id": dataset_id, "bbox": BBOX, "period": period}
            query.update(base_extra)
            products = page_search(client, query)
            download_matches(products, out)

    print("\n✅ All done. Files saved under:", OUT_DIR.resolve())

In [ ]:
"""
Copernicus CLMS HRL Tree Cover Downloader (HDA API)
- Loops over datasets: TCD, TCD_CONF, TCD_CHANGE, TCD_CHANGE_CONF
- Fetches ALL years/periods for Kosovo bbox
- Downloads all matching tiles into a clear folder structure
- Skips files that already exist
- Optional: merges per (dataset, year/period) after download using GDAL (commented out)

Requirements:
    conda activate SLK_deforestation
    conda install -c conda-forge hda requests tqdm gdal

Credentials:
    Create ~/.hdarc once, or let the official HDA notebook do it for you:
    {
      "user": "YOUR_EU_LOGIN_USERNAME",
      "password": "YOUR_EU_LOGIN_PASSWORD"
    }
"""

import os
import sys
from pathlib import Path
from typing import Dict, List, Tuple

# Third-party
try:
    from hda import Client, Configuration
except Exception as e:
    print("ERROR: The 'hda' package is required. Install with: conda install -c conda-forge hda")
    raise

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# ---------- CONFIG ----------

# --- Direct HDA credentials (WeKEO account) ---
# HDA_USER = "user"
# HDA_PASSWORD = "pass"

# Use WeKEO endpoint (instead of default EEA one)
# HDA_URL = "https://wekeo-broker.apps.mercator.dpi.wekeo.eu/hda-broker"

# Kosovo bbox in EPSG:4326 [minLon, minLat, maxLon, maxLat]
BBOX = [20.0, 41.8, 21.8, 43.3]

# Output root
OUT_DIR = Path("/Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/.shortcut-targets-by-id/1p4PDClIbV5Br-EkGzuvhH5WVU1BudYkH/SLK CorrelAid 2025/Copernicus/batchimport")

# Datasets and their temporal parameters
# For CHANGE products, HRL uses periods like "2012-2015", "2015-2018", "2018-2023".
DATASETS: Dict[str, Dict] = {
    # Tree Cover Density (numeric % per pixel)
    "EO:EEA:DAT:HRL:TCF": {
        "label": "TCD",
        "years": ["2012", "2015", "2018", "2023"],
        "extra": {"resolution": "10m", "productType": "Tree Cover Density"}
    },
    # Confidence layer for TCD
    "EO:EEA:DAT:HRL:TCF_CONF": {
        "label": "TCD_CONF",
        "years": ["2012", "2015", "2018", "2023"],
        "extra": {"resolution": "10m", "productType": "Tree Cover Density Confidence"}
    },
    # Tree Cover Presence Change
    "EO:EEA:DAT:HRL:TCF_CHANGE": {
        "label": "TCD_CHANGE",
        "periods": ["2012-2015", "2015-2018", "2018-2023"],
        "extra": {"resolution": "10m", "productType": "Tree Cover Presence Change"}
    },
    # Confidence layer for Change
    "EO:EEA:DAT:HRL:TCF_CHANGE_CONF": {
        "label": "TCD_CHANGE_CONF",
        "periods": ["2012-2015", "2015-2018", "2018-2023"],
        "extra": {"resolution": "10m", "productType": "Tree Cover Presence Change Confidence"}
    },
}

# HDA pagination
ITEMS_PER_PAGE = 200
START_INDEX = 0

# Toggle automatic merge with GDAL after download (per group)
RUN_GDAL_MERGE = False  # set to True if you want automatic mosaics
GDAL_OUTPUT_CRS = "EPSG:3857"  # used only in the example warp step below

# ---------- END CONFIG ----------


def ensure_outdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)


def hda_client_or_exit() -> Client:
    hda_client = Client()
    return hda_client
    # hdarc = Path(Path.home()/'.hdarc')
    # try:
    #     client = Client(config=hdarc)
    #     client.metadata(dataset_id="EO:EEA:DAT:HRL:TCF")  # test connection
    #     print(f"✅ Logged in as {client.user}")
    #     return client
    # except Exception as e:
    #     print("❌ Failed to initialize HDA Client.")
    #     raise




def page_search(client: Client, query: dict) -> list:
    """
    Notebook-style search: single call, returns list of products.
    """
    print(f"Searching {query['dataset_id']} with filter { {k:v for k,v in query.items() if k not in ('dataset_id','bbox')} }")
    results = client.search(
        dataset_id=query["dataset_id"],
        bbox=query["bbox"],
        productType=query.get("productType"),
        resolution=query.get("resolution"),
        year=query.get("year"),
        period=query.get("period"),
    )
    print(f"  → {len(results)} products found")
    return results


def download_matches(matches: list, out_dir: Path):
    ensure_outdir(out_dir)
    # Each "match" is a hda.model.product.Product (with .download(path))
    iterator = matches
    if tqdm is not None:
        iterator = tqdm(matches, desc=f"Downloading to {out_dir}", unit="tile")
    for prd in iterator:
        # The product usually has .title or .id which maps to filename inside download
        # The .download() method will use the suggested filename from server.
        try:
            prd.download(str(out_dir))
        except Exception as e:
            print(f"Download failed for {getattr(prd, 'id', 'unknown')}: {e}")


def gdal_merge_group(in_dir: Path, pattern: str, out_path: Path):
    """
    Optional: mosaic GeoTIFFs using gdal_merge.py. Requires GDAL in PATH.
    """
    import subprocess, glob, shlex
    files = sorted(glob.glob(str(in_dir / pattern)))
    if not files:
        print(f"No files to merge for pattern {pattern} in {in_dir}")
        return
    ensure_outdir(out_path.parent)
    cmd = ["gdal_merge.py", "-o", str(out_path), "-of", "GTiff", "-a_nodata", "0"] + files
    print("Merging:", out_path.name, f"({len(files)} tiles)")
    subprocess.run(cmd, check=True)


def main():
    ensure_outdir(OUT_DIR)
    client = hda_client_or_exit()

    for dataset_id, cfg in DATASETS.items():
        label = cfg["label"]
        base_extra = cfg.get("extra", {})

        # --- yearly datasets
        for year in cfg.get("years", []):
            print(f"\n=== {label} {year} ===")
            out = OUT_DIR / label / year
            ensure_outdir(out)
            query = {"dataset_id": dataset_id, "bbox": BBOX, "year": year}
            query.update(base_extra)
            products = page_search(client, query)
            for p in tqdm(products, desc=f"{label} {year}", unit="tile"):
                try:
                    p.download(str(out))
                except Exception as e:
                    print(f"Download failed for {getattr(p,'id',p)}: {e}")
            # if RUN_GDAL_MERGE:
            #     merged = OUT_DIR / "merged" / f"{label}_{year}_merged.tif"
            #     gdal_merge_group(out, "*.tif", merged)

        # --- period datasets
        for period in cfg.get("periods", []):
            print(f"\n=== {label} {period} ===")
            out = OUT_DIR / label / period
            ensure_outdir(out)
            query = {"dataset_id": dataset_id, "bbox": BBOX, "period": period}
            query.update(base_extra)
            products = page_search(client, query)
            for p in tqdm(products, desc=f"{label} {period}", unit="tile"):
                try:
                    p.download(str(out))
                except Exception as e:
                    print(f"Download failed for {getattr(p,'id',p)}: {e}")
            # if RUN_GDAL_MERGE:
            #     merged = OUT_DIR / "merged" / f"{label}_{period}_merged.tif"
            #     gdal_merge_group(out, "*.tif", merged)

    print("\n✅ All done. Files saved under:", OUT_DIR.resolve())



if __name__ == "__main__":
    main()
